In [ ]:
import tkinter as tk
import random
import pickle
import numpy as np

# ── constants
ROWS, COLS = 6, 7
CELL_SIZE = 80

# ── load model
with open("dataset/model.pkl", "rb") as f:
    model = pickle.load(f)

COLUMNS = [
    "a1","a2","a3","a4","a5","a6",
    "b1","b2","b3","b4","b5","b6",
    "c1","c2","c3","c4","c5","c6",
    "d1","d2","d3","d4","d5","d6",
    "e1","e2","e3","e4","e5","e6",
    "f1","f2","f3","f4","f5","f6",
    "g1","g2","g3","g4","g5","g6",
]

# ── board functions
def drop_piece(board, col, player):
    for r in range(ROWS-1, -1, -1):
        if board[r][col] == 0:
            board[r][col] = player
            return True
    return False

def check_winner(board, player):
    for r in range(ROWS):
        for c in range(COLS-3):
            if all(board[r][c+i]==player for i in range(4)): return True
    for r in range(ROWS-3):
        for c in range(COLS):
            if all(board[r+i][c]==player for i in range(4)): return True
    for r in range(ROWS-3):
        for c in range(COLS-3):
            if all(board[r+i][c+i]==player for i in range(4)): return True
    for r in range(3, ROWS):
        for c in range(COLS-3):
            if all(board[r-i][c+i]==player for i in range(4)): return True
    return False

def opponent_move(board):
    valid_cols = [c for c in range(COLS) if board[0][c] == 0]
    if valid_cols:
        col = random.choice(valid_cols)
        drop_piece(board, col, 2)

def board_to_model_input(board):
    flat = []
    col_letters = "abcdefg"
    for ci, col in enumerate(col_letters):
        for ri in range(1, ROWS+1):
            flat.append(board[ROWS-ri][ci])
    return np.array(flat).reshape(1, -1)

def predict_winner(board):
    X = board_to_model_input(board)
    pred = model.predict(X)[0]
    labels = {0: "Draw", 1: "Loss for X", 2: "Win for X"}
    return labels[pred]

# ── GUI
window = tk.Tk()
window.title("Connect 4 — AI Edition")
window.configure(bg="#1a1a2e")
window.resizable(False, False)

canvas = tk.Canvas(window, width=COLS*CELL_SIZE, height=ROWS*CELL_SIZE,
                   bg="#1a1a2e", highlightthickness=0)
canvas.pack()
legend_frame = tk.Frame(window, bg="#1a1a2e")
legend_frame.pack(pady=5)

tk.Canvas(legend_frame, width=20, height=20, bg="#ff4757", highlightthickness=0).grid(row=0, column=0, padx=5)
tk.Label(legend_frame, text="You (X)", bg="#1a1a2e", fg="#ff4757", font=("Courier", 11)).grid(row=0, column=1, padx=5)

tk.Canvas(legend_frame, width=20, height=20, bg="#ffa502", highlightthickness=0).grid(row=0, column=2, padx=15)
tk.Label(legend_frame, text="Opponent (O)", bg="#1a1a2e", fg="#ffa502", font=("Courier", 11)).grid(row=0, column=3, padx=5)
status_label = tk.Label(window, text="Your turn! Click a column.",
                        font=("Courier", 12), bg="#1a1a2e", fg="#ffffff", pady=10)
status_label.pack()

# ── draw
def draw_board(board):
    canvas.delete("all")
    for r in range(ROWS):
        for c in range(COLS):
            x = c * CELL_SIZE + CELL_SIZE // 2
            y = r * CELL_SIZE + CELL_SIZE // 2
            color = "#2a2a4a"
            if board[r][c] == 1: color = "#ff4757"
            if board[r][c] == 2: color = "#ffa502"
            canvas.create_oval(x-30, y-30, x+30, y+30,
                               fill=color, outline="#4a4a8a", width=2)

# ── click handler
def get_col_from_click(x):
    return x // CELL_SIZE

def handle_click(event):
    col = get_col_from_click(event.x)
    if not drop_piece(board, col, 1): return
    draw_board(board)
    if check_winner(board, 1):
        status_label.config(text="🎉 You Win!")
        canvas.unbind("<Button-1>")
        return
    opponent_move(board)
    draw_board(board)
    if check_winner(board, 2):
        status_label.config(text="😔 Opponent Wins!")
        canvas.unbind("<Button-1>")
        return
    prediction = predict_winner(board)
    status_label.config(text=f"AI says: {prediction}")

board = [[0]*COLS for _ in range(ROWS)]
draw_board(board)
canvas.bind("<Button-1>", handle_click)

window.mainloop()